# FER-2013: M1 custom CNN and M2 ResNet-18

This is the lab's local training and evaluation notebook. Read the architecture
and training explanations alongside the small helper functions in `src/`.
The implementation follows the lab PDF; M1 follows the later supplied image.

Select the `C:\Users\ADMIN\ai_venv` Jupyter kernel, install requirements,
and extract [FER-2013](https://www.kaggle.com/datasets/msambare/fer2013) before running.

In [ ]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the repository root or in notebooks/.
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((p for p in candidates if (p / "src" / "model.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in this repository or its notebooks/ directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)

In [ ]:
import json
import shutil
import torch
from IPython.display import Image, display

from src.data import CLASS_NAMES, build_data
from src.model import build_model, count_parameters
from dataclasses import asdict
from src.train import seed_everything
from src.experiments import accuracy_recipes, run_accuracy_experiments
from src.evaluate import plot_class_distribution, evaluate_checkpoint, compare_models
from src.report import build_report

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data" / "fer2013"  # Contains train/ and test/.
RUN_NAME = "fer2013_accuracy_v2"  # Change for a new experiment; completed candidates can be reused.
RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
OUTPUT_ROOT = RUN_ROOT / "outputs"
WEIGHTS_DIR = RUN_ROOT / "weights"
REPORT_PATH = RUN_ROOT / "report.pdf"
BATCH_SIZE = 128  # Matches the first Kaggle run; reduce if memory is insufficient.
NUM_WORKERS = 0
VAL_FRACTION = 0.10
SEED = 42
# Optional: old outputs/ containing cnn/ and transfer/ with best.pt + manifests.
# E.g. PROJECT_ROOT / "kaggle_outputs" / "run1_1025" / "outputs"
BASELINE_RESULTS_ROOT = None

## 1. Inspect the data and the split

Both models use the same stratified train/validation split. Validation is held
out of the original `train/` directory, never from `test/`. The class weights
are computed **only from training labels**: square-root inverse frequency,
normalized to mean one. This gives rare classes more influence without the
very large weights of full inverse frequency.

Training adds mild flips/affine augmentation. Every split uses grayscale
48x48 and normalization `(pixel / 255 - 0.5) / 0.5`. The app reuses this code.

The initial plot shows the fixed baseline split. Each candidate rebuilds its training loader
with its own augmentation profile and the same split seed. Loss weights remain
square-root inverse-frequency; we do not stack them with oversampling.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)
data = build_data(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  val_fraction=VAL_FRACTION, seed=SEED)
counts = plot_class_distribution(data, OUTPUT_ROOT / "class_distribution.png")
data.save_manifest(OUTPUT_ROOT / "split_manifest.json")
print("Resolved data directory:", data.root)
print(json.dumps(counts, indent=2))
print("Class order:", CLASS_NAMES)
print("Training class weights:", data.class_weights.tolist())
display(Image(filename=str(OUTPUT_ROOT / "class_distribution.png")))

## 2. Inspect M1 and M2

**M1 matches the supplied architecture.** Each block contains one padded 3x3
convolution, BatchNorm, ReLU, 2x2 max pool, and ordinary dropout. Channels:
32, 64, 128, 256; dropout: 0.10, 0.15, 0.20, 0.25. Its spatial sizes are
24, 12, 6, 3, so flattening produces 2304 features. The head is
Linear(2304,256), BatchNorm, ReLU, Dropout(0.50), Linear(256,7).

**M2 is ImageNet-pretrained ResNet-18.** The first convolution accepts one
grayscale channel; its RGB filters are summed, its stride becomes 1, and the
initial max pool is removed to retain detail at 48x48. The final layer becomes
Dropout(0.30) + Linear(512,7). All layers are fine-tuned, with a smaller learning
rate in the backbone. The shared FER normalization differs from stock ImageNet
preprocessing; fine-tuning adapts the network to that change.

Both networks return **logits**, not softmax, because CrossEntropyLoss applies
log-softmax itself. Inference applies softmax once to display confidence.

In [ ]:
torch.set_num_threads(4)
for name in ["cnn", "transfer"]:
    # Architecture inspection does not download pretrained weights.
    model = build_model(name, pretrained=False).eval()
    with torch.inference_mode():
        scores = model(torch.zeros(2, 1, 48, 48))
    print(f"{name}: {count_parameters(model):,} parameters; output {tuple(scores.shape)}")
    del model

## 3. Accuracy-first candidate comparison

The first run reached 65.99% test accuracy (M1) and 67.69% (M2). The next-run
candidates are hypotheses; improvements are measured on **validation**, never
assumed from those test results.

- M1: original baseline versus baseline + brightness/contrast jitter (0.15 each).
- M2: original baseline, lighting augmentation, and a regularized candidate with
  lighting + small erasing (p=0.10, area 2-6%), weight decay 5e-4 and backbone LR 5e-5.
- Both retain the current moderate class weights and label smoothing 0.05.
- Each fit selects by validation accuracy; macro F1 breaks exact ties. The same
  rule chooses among candidates. If revised candidates lose, baseline wins.

M1 keeps its exact supplied architecture and 80-epoch limit. M2 candidates each
use the first run's 20-epoch budget. The regularized candidate combines changes
for a practical accuracy search; it is not a one-factor causal ablation.
Five complete fits take longer than the first run. You can reuse old baseline
checkpoints through BASELINE_RESULTS_ROOT; matching train/validation manifests
and training configs are required. The saved validation predictions let you
inspect per-class performance without repeatedly opening test results.

AMP is used during GPU training; validation and deployment use float32.
Training curves show plain cross entropy. The weighted/smoothed objective is
saved separately. Completed matching candidates are cached. An interrupted fit
cannot resume its optimizer; choose a new RUN_NAME to retrain it.

In [ ]:
cnn_recipes = accuracy_recipes("cnn", seed=SEED)
transfer_recipes = accuracy_recipes("transfer", seed=SEED)
for model_name, recipes in [("cnn", cnn_recipes), ("transfer", transfer_recipes)]:
    print("\n", model_name.upper())
    for candidate_name, config in recipes.items():
        print(candidate_name, asdict(config))
# M2 recipes use pretrained=True; ImageNet weights download on first training use.

## 4. Compare M1 recipes

Train/reuse the baseline and lighting candidates. Only validation chooses the exported checkpoint.
Reduce BATCH_SIZE if memory is insufficient, before constructing loaders.

In [ ]:
cnn_run = run_accuracy_experiments(
    "cnn", DATA_ROOT, OUTPUT_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    val_fraction=VAL_FRACTION, split_seed=SEED, seed=SEED,
    baseline_results_root=BASELINE_RESULTS_ROOT, recipes=cnn_recipes,
)
print("Selected M1:", cnn_run["selected_candidate"])
print("Checkpoint:", cnn_run["checkpoint"])

## 5. Compare M2 fine-tuning recipes

Baseline, lighting, and regularized candidates all use ImageNet initialization when trained.
Each candidate starts with newly seeded loaders and the same validation split.

In [ ]:
transfer_run = run_accuracy_experiments(
    "transfer", DATA_ROOT, OUTPUT_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    val_fraction=VAL_FRACTION, split_seed=SEED, seed=SEED,
    baseline_results_root=BASELINE_RESULTS_ROOT, recipes=transfer_recipes,
)
print("Selected M2:", transfer_run["selected_candidate"])
print("Checkpoint:", transfer_run["checkpoint"])
# The final test loader is deterministic and shared across all selected recipes.
data = build_data(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                  val_fraction=VAL_FRACTION, seed=SEED)

## 6. Final test evaluation and M3 comparison

Evaluate only the selected validation winner for each model. Candidate selection is already finished. Export accuracy,
macro F1, parameter counts, and CPU forward FPS, along with the full confusion
matrix and per-class report. CPU timing is **model-only**, batch one, float32,
four threads; it excludes face detection, preprocessing and camera I/O.
The application's displayed FPS measures capture + processing separately.

The lab targets (60% M1, 65% M2) are checked against measured test accuracy;
the code does not assume they have been achieved.

In [ ]:
all_metrics = []
for name in ["cnn", "transfer"]:
    metrics = evaluate_checkpoint(
        OUTPUT_ROOT / name / "best.pt", data, device="auto",
    )
    all_metrics.append(metrics)
    shutil.copy2(OUTPUT_ROOT / name / "best.pt", WEIGHTS_DIR / f"{name}_best.pt")
    print(json.dumps(metrics, indent=2))

comparison = compare_models(all_metrics, OUTPUT_ROOT)
print(f"{'Model':<10} {'Accuracy':>10} {'Macro F1':>10} {'Parameters':>12} {'CPU FPS':>10}")
for row in comparison:
    print(f"{row['model']:<10} {row['test_accuracy']:>10.2%} "
          f"{row['test_macro_f1']:>10.4f} {row['parameters']:>12,} {row['cpu_model_fps']:>10.1f}")

## 7. M4 and M5: inspect plots and mistakes

Each model exports up to ten errors, covering different true classes when
possible. The probability-based comments in `error_analysis.md` are a starting
point: inspect the faces and add image-specific observations about pose,
occlusion, lighting or ambiguous expressions. They are not invented visual
explanations. If fewer than ten mistakes exist, all available mistakes are shown.

In [ ]:
for name in ["cnn", "transfer"]:
    print("\n", name.upper())
    for filename in ["training_curves.png", "confusion_matrix.png", "misclassified.png"]:
        path = OUTPUT_ROOT / name / filename
        if path.exists():
            display(Image(filename=str(path)))
    print((OUTPUT_ROOT / name / "error_analysis.md").read_text(encoding="utf-8"))

## 8. Generate the six-page report

This uses actual saved results. Until you provide a phone-camera screenshot
and finish your visual error analysis, treat the PDF as a draft. Run the app
locally and press **s** to save `outputs/demo_screenshot.jpg`, then regenerate
the report. Edit `src/report.py` to incorporate your own written analysis.

In [ ]:
screenshot = OUTPUT_ROOT / "demo_screenshot.jpg"
report_path = build_report(
    OUTPUT_ROOT, REPORT_PATH,
    screenshot=screenshot if screenshot.exists() else None,
    author="YOUR NAME / STUDENT ID",
)
print("Report:", report_path)
print("Weights:", WEIGHTS_DIR)

## Before submission

Keep all full FER training/evaluation outputs in this notebook. The PDF asks
for an **executed** `notebooks/train.ipynb`; running only the Kaggle notebook
does not execute this local file. If you train on Kaggle, also upload
`notebooks/train.ipynb` as a Kaggle notebook and run it against the same clone
(adjust its setup/data/output paths), or run the local notebook with a GPU.

Copy the best checkpoints into `weights/`, finish the report, and test:

```text
python src/app.py --source http://<ip>:8080/video --model cnn
python src/app.py --source test.jpg --model transfer
```

Record actual CPU realtime FPS on your laptop, with the phone connected to the
same Wi-Fi network. Source URLs and checkpoint paths live in `config.yaml`.

This run keeps its checkpoints in RUN_ROOT / "weights". After reviewing the results,
copy the two *_best.pt files into the project-level weights/ folder used by config.yaml.
Run the app locally, save the phone demo screenshot, and regenerate the report.